# Client Command Assessment

The purpose of this notebook is to analyze the given datasets and find ["who is most likely to purchase a vehicle."](https://github.com/SummitResources/data_engineer_technical_assessment/blob/master/README.md)

The time limit is two hours. Start time is:
**8:10 PM Jan 30, 2020**

I will begin by reading in the necessary libraries and exploring the data. After an initial preview of the csv files, these are my initial observations:
* It seems as though several of the included files are repetitions of the same schema (`'online_behavior_0'`, `'online_behavior_1'`, etc.).
* These csv files have three columns: 
    * `'activity_time'` - a timestamp
    * `'list_id'` - an activity code
    * `'email'` - an encoded personal email address
* `'emails.csv'` contains personal identifiers:
    * `'email'` - an encoded personal email address
    * `'first_name'`, `'last_name'`
    * `'address'`, `'city'`, `'state'`, `'zip'`, `'zip4'` - address information
* `'online_behavior_taxonomy.csv'` contains information on the activity codes:
    * `'list_id'` - the same code from the `'online_behavior'` files
    * `'list_name'` - a string with information about the `'list_id'`
    
After this analysis, it seems the best approach to combine these datasets into useful information would be:
* Clean and remove unnecessary data from the taxonomy file
* Split the `'list_name'` field into useful categories (make, model, etc.)
* Concatenate and dedupe the `'online_behavior'` csv files
* Match each record to a `'list_name'` and name/address
* Match each record to a taxonomy record
* Group by email address and find summary statistics for each individual
* Use summary statistics to rank best prospects
* If time allows, use geographical email to break down into local area lists

In [ ]:
# importing libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

## Cleaning Taxonomy File

In [ ]:
# reading in taxonomy file
tax = pd.read_csv('online_behavior_taxonomy.csv')
tax.info()

In [ ]:
# exploring taxonomy file
tax.head(20)

In [ ]:
# exploring unique values in dataset
print(len(tax['list_name'].unique()))

This list consists of 572 codes for web activities. Only one seems to be duplicated.

Values containing "Client Command Target Lists" do not help in this analysis, since their meaning is unknown. The other data, however, is descriptive enough to help us determine what make, model, price range, and 'context' the buyers are looking for.

I will remove the unnecessary data from this index, and use the index to remove unnecessary records from the main datasets.

In [ ]:
# removing all "Client Command Target Lists" records
tax_clean = tax[~tax.list_name.str.contains("Client Command Target")]
# remove duplicates from list
tax_clean = tax_clean.drop_duplicates()
print(len(tax_clean))

This eliminated 230 values from the list.

Next, I will split the `'list_name'` category into separate columns to help create more useable column data. I will split using the " > " between each text value. This should leave separate word/phrases without spaces.

**Note:** there are some values which are not make or model information. We will keep these in the Make/Model category for now and move them to new columns in the next step.

In [ ]:
# creating new columns and splitting list name
tax_clean[['category','make','model']] = tax_clean['list_name'].str.split(' > ', expand=True)
# dropping old column data
tax_clean.drop(['list_name'], axis=1, inplace=True)
tax_clean.head(20)

I now need to clean up the `'make'` column, since it has other non-make values. For any row whose category is not "Makes," I will move the "Make" value to a new column.

In [ ]:
# creating a new column with null values
tax_clean['descriptor'] = np.NaN

# creating a mask to check for non-make values
tax_clean.loc[tax_clean['category']!='Makes', 'descriptor'] = tax_clean['make']
# removing non-make values from make column
tax_clean.loc[tax_clean['category']!='Makes', 'make'] = np.NaN

In [ ]:
tax_clean.head(20)

It looks as though this data is now clean and ready to be combined with the `'online_behavior'` csv files.

## Concatenate and dedupe online_behavior files

In [ ]:
# read files into dataframes
ob_0 = pd.read_csv('online_behavior_0.csv')
ob_1 = pd.read_csv('online_behavior_1.csv')
ob_2 = pd.read_csv('online_behavior_2.csv')
ob_3 = pd.read_csv('online_behavior_3.csv')
ob_4 = pd.read_csv('online_behavior_4.csv')
ob_5 = pd.read_csv('online_behavior_5.csv')
# note - if time permitted, this could be standardized
# into a function with increasing numerical values

In [ ]:
# inspecting dataframes to ensure continuity
ob_0.head()

In [ ]:
ob_1.head()

In [ ]:
ob_2.head()

In [ ]:
ob_3.head()

In [ ]:
ob_4.head()

In [ ]:
ob_5.head()

All five dataframes have the same structure and can be concatenated together.

In [ ]:
# concatenating dataframes into one master dataframe
online_behavior = pd.concat([ob_0, ob_1, ob_2, ob_3, ob_4, ob_5], ignore_index=True)
online_behavior.describe()

In [ ]:
# remove duplicates from master dataframe
online_behavior = online_behavior.drop_duplicates()

In [ ]:
online_behavior.info()

The final master dataframe has over 1 million rows, none of which are null.

## Matching records to name and address

We can now match rows in the master dataframe to names and addresses in the `'emails'` file, using email as a primary key. To do this, we will use a merge, bringing data from the `'emails'` file in to match rows in the master dataframe.

In [ ]:
# reading in emails file
emails = pd.read_csv('emails.csv')
emails.head()

In [ ]:
emails.info()

In [ ]:
ob_with_names = online_behavior.merge(emails, how='left', on="email")

In [ ]:
ob_with_names.info()

In [ ]:
ob_with_names.head()

It seems as though many of the records in the main dataframe do not have a match in the `'emails'` file. These will be "unknown" individuals, identified only by email address and behavior.

## Match each record to a taxonomy record

Now, I need to bring in the web behavior to each record, merging in the `'tax_clean'` dataframe from above. The method will be the same as the last merge.

In [ ]:
# merging taxonomy data into master dataframe
master = ob_with_names.merge(tax_clean, how='left', on="list_id")

In [ ]:
master.info()

Remember from above, we removed all values which did not have a useable `'category'` value. We can now eliminate all rows whose values for `'category'` is null.

In [ ]:
# dropping all rows with null values in category
master = master.dropna(axis=0, subset=['category'])
master.info()

Now the lengths of the `'email'` and `'category'` columns match, so there are no values with unuseable `'list_id'` data.

## Get number of occurrences by email

Now I can groupby `'email'` and count the number of entries for each unique person in the dataframe. This could be a good indicator of desire to purchase.

In [ ]:
# creating groupby object for summary count
master_group = master.groupby('email').agg('count')

In [ ]:
master_group.head()

There is obviously a problematic `'email'` value, with zeros for an email value. These rows should be dropped from the master dataframe, and the groupby recalculated.

In [ ]:
# removing bad values from master dataframe
master = master[master.email != '00000000000000000000000000000000']

# recreating groupby object for summary count
master_group = master.groupby('email').agg('count')

In [ ]:
master_group.sort_values(by='list_id', ascending=False, inplace=True)
master_group.head()

In [ ]:
# exploring summary statistics for the count of individuals
master_group['list_id'].describe()

In [ ]:
master_group['list_id'].plot.box()

The top instance in this grouping seems suspect, and may be an outlier which needs to be removed. But it is possible this person is not an outlier, so we will keep it in the master dataframe and continue to explore.

## List One: most active prospects

Using this groupby dataframe, we can create a new list of prospects sorted by number of instances. I will create this sorted dataframe, which could eventually be exported to a new file or database table.

In [ ]:
# creating new dataframe with only email and count columns
# saving index of master_group to new column
master_group.reset_index(inplace=True)
# list of columns to keep
cols = ['email', 'list_id']
most_active = master_group[cols]

In [ ]:
# merging in name and address data
most_active_final = most_active.merge(emails, how='left', on="email")
most_active_final.info()

This new dataframe is available with a listing of the most active users, including geographical data. This could easily be broken down into geographical location by state or zip code, for specific dealerships.

# Creating make and car-specific counts

Now, I can return to the master datafame and one-hot encode the `'category'`, `'make'` and `'model'` columns. By doing so, it will allow me to perform a new groupby with a sum of each column, showing the interest of each person in particular vehicles.

Once this is done, I will merge this back in with the dataframe above, to create a record for each individual with activity level and relative interest in each categorymake, model and descriptor.

Because I will eventually merge this back in, I can delete the duplicate columns to keep the dataframe as simple as possible.

In [ ]:
# refamiliarizing myself with master
master.head()

In [ ]:
# list of columns to keep
cols = ['activity_time','email','category','make','model','descriptor']
# keeping only these columns
master_detail = master[cols]

In [ ]:
# creating list of columns to get dummies
dummy_cols = ['make','model','descriptor']
# creating dummy valus
master_detail_dummies = pd.get_dummies(master_detail[dummy_cols])

In [ ]:
# creating dataframe with unused columns to merge
cols = ['activity_time', 'email']
master_emails = master[cols]
# concatenating two dataframes to include email information
master_info = pd.concat([master_emails, master_detail_dummies], axis=1)

In [ ]:
master_info.head()

We now have information for each unique instance across numerous makes, models, and categories. If we now groupby email and aggregate by sum, we will get a value for each individual's interest level in these variables.

In [ ]:
# creating groupby object grouping by email
master_info_agg = master_info.groupby('email').agg('sum')

Now, I can merge this info with aggregate sums for interest level with the `'most_active_final'` dataframe to create a final dataframe that contains useable info about prospect behavior.

In [ ]:
# merging both into final dataframe
prospect_info = most_active_final.merge(master_info_agg, how='left', on="email")

In [ ]:
# renaming list_id column to represent activity
prospect_info = prospect_info.rename(columns={'list_id': 'engagements'})

In [ ]:
prospect_info.describe()

In [ ]:
prospect_info.head()

In [ ]:
# Saving this prospect dataframe to a file for future exploration
prospect_info.to_csv('prospect_info.csv')

# Results

The result of this project is a dataset with unified data on user behavior. To use this file, a dealer could filter based on geographical area, sort by number of engagements overall, and then drill down to the number of engagements based on make, model, or other descriptors of interest.

Finish time is: **10:11 PM Jan 30, 2020**